In [ ]:
import os
import sys
import json
import csv
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from itertools import permutations

def parse_paths():
    if len(sys.argv) > 2 and not sys.argv[1].startswith('-f'):
        return Path(sys.argv[1]), Path(sys.argv[2])
    cands = [
        Path('./dataset/public'),
        Path('/kaggle/input/datasets/devansh66/logical-bundle'),
        Path('/kaggle/input/logical-bundle'),
        Path('/kaggle/input'),
        Path('.')
    ]
    pub = next((c for c in cands if (c / 'tasks.json').exists()), Path('./dataset/public'))
    out = Path('./working/submission.csv')
    os.makedirs(out.parent, exist_ok=True)
    return pub, out

PUB, OUT = parse_paths()

catalog = json.load(open(PUB / 'catalog.json', encoding='utf-8'))
tasks = json.load(open(PUB / 'tasks.json', encoding='utf-8'))

by_id = {c['item_id']: c for c in catalog}

_idx = {}
for c in catalog:
    for key in [
        (c['category'],),
        (c['category'], c['color'], None),
        (c['category'], None, c['brand']),
        (c['category'], c['color'], c['brand']),
    ]:
        _idx.setdefault(key, []).append(c)

for k in _idx:
    _idx[k].sort(key=lambda x: (x['price'], x['item_id']))

def get_candidates(cat, color=None, brand=None):
    return _idx.get((cat, color, brand), [])

def extract_focal(items, mode, focal_cat=None):
    if not items:
        return None
    if mode == 'max':
        return max(items, key=lambda x: (x['price'], -x['item_id']))
    elif mode == 'min':
        return min(items, key=lambda x: (x['price'], x['item_id']))
    elif mode == 'category' and focal_cat:
        sub = [x for x in items if x['category'] == focal_cat]
        if not sub:
            return None
        return max(sub, key=lambda x: (x['price'], -x['item_id']))
    return None

def closest_item(cands, target_price, exclude=None):
    if exclude is None:
        exclude = set()
    valid = [c for c in cands if c['item_id'] not in exclude]
    if not valid:
        return None
    return min(valid, key=lambda x: (abs(x['price'] - target_price), x['price'], x['item_id']))

def select_extremum(cands, minimize=False, exclude=None):
    if exclude is None:
        exclude = set()
    valid = [c for c in cands if c['item_id'] not in exclude]
    if not valid:
        return None
    if minimize:
        return min(valid, key=lambda x: (x['price'], x['item_id']))
    else:
        return max(valid, key=lambda x: (x['price'], -x['item_id']))

def generate_hypotheses(out_cat):
    hyps = []
    for c_src in ['none', 'cart_majority', 'cart_focal_color', 'const']:
        for b_src in ['none', 'cart_majority', 'cart_focal_brand', 'const']:
            for p_rule in ['max', 'min']:
                hyps.append({
                    'type': 'extremum',
                    'color_src': c_src,
                    'brand_src': b_src,
                    'price_rule': p_rule,
                    'focal_type': 'max',
                })
    for f_type in ['max', 'min', 'category']:
        f_cats = [out_cat] if f_type == 'category' else [None]
        for fcat in f_cats:
            for mult in [1.0, 0.5, 2.0, 1.5, 0.8]:
                for c_src in ['none', 'cart_majority', 'cart_focal_color']:
                    for b_src in ['none', 'cart_majority', 'cart_focal_brand']:
                        hyps.append({
                            'type': 'closest',
                            'focal_type': f_type,
                            'focal_cat': fcat,
                            'mult': mult,
                            'color_src': c_src,
                            'brand_src': b_src
                        })
    return hyps

def eval_hypothesis(hyp, inp_items, true_out, out_cat):
    color, brand = None, None
    focal = extract_focal(inp_items, hyp.get('focal_type', 'max'), hyp.get('focal_cat'))
    
    c_src = hyp.get('color_src')
    if c_src == 'cart_majority':
        color = Counter(x['color'] for x in inp_items).most_common(1)[0][0]
    elif c_src == 'cart_focal_color' and focal:
        color = focal['color']
    elif c_src == 'const':
        color = true_out['color']
        
    b_src = hyp.get('brand_src')
    if b_src == 'cart_majority':
        brand = Counter(x['brand'] for x in inp_items).most_common(1)[0][0]
    elif b_src == 'cart_focal_brand' and focal:
        brand = focal['brand']
    elif b_src == 'const':
        brand = true_out['brand']
        
    cands = get_candidates(out_cat, color, brand)
    if not cands:
        cands = get_candidates(out_cat)
    if not cands:
        return None
        
    exclude = set(x['item_id'] for x in inp_items)
    
    if hyp['type'] == 'extremum':
        return select_extremum(cands, minimize=(hyp['price_rule'] == 'min'), exclude=exclude)
    elif hyp['type'] == 'closest' and focal:
        t_price = focal['price'] * hyp['mult']
        return closest_item(cands, t_price, exclude=exclude)
    return None

def find_best_hypothesis(slot_data, out_cat):
    hyps = generate_hypotheses(out_cat)
    best_hyp = None
    best_score = -1
    
    for hyp in hyps:
        score = 0
        for inp_items, true_out in slot_data:
            pred = eval_hypothesis(hyp, inp_items, true_out, out_cat)
            if pred and pred['item_id'] == true_out['item_id']:
                score += 1
        if score > best_score:
            best_score = score
            best_hyp = hyp
            if best_score == len(slot_data):
                break
                
    if best_score <= 0:
        best_hyp = {
            'type': 'extremum',
            'color_src': 'none',
            'brand_src': 'none',
            'price_rule': 'max'
        }
        
    return best_hyp, best_score

def apply_hypothesis(hyp, inp_items, out_cat, exclude=None):
    if exclude is None:
        exclude = set()
    color, brand = None, None
    focal = extract_focal(inp_items, hyp.get('focal_type', 'max'), hyp.get('focal_cat'))
    
    c_src = hyp.get('color_src')
    if c_src == 'cart_majority':
        color = Counter(x['color'] for x in inp_items).most_common(1)[0][0]
    elif c_src == 'cart_focal_color' and focal:
        color = focal['color']
        
    b_src = hyp.get('brand_src')
    if b_src == 'cart_majority':
        brand = Counter(x['brand'] for x in inp_items).most_common(1)[0][0]
    elif b_src == 'cart_focal_brand' and focal:
        brand = focal['brand']
        
    cands = get_candidates(out_cat, color, brand)
    valid = [c for c in cands if c['item_id'] not in exclude]
    if not valid:
        cands = get_candidates(out_cat)
        valid = [c for c in cands if c['item_id'] not in exclude]
    if not valid:
        return None
        
    if hyp['type'] == 'extremum':
        return select_extremum(valid, minimize=(hyp['price_rule'] == 'min'), exclude=exclude)
    elif hyp['type'] == 'closest' and focal:
        t_price = focal['price'] * hyp.get('mult', 1.0)
        return closest_item(valid, t_price, exclude=exclude)
        
    return valid[0]

def align_slots(train_exs):
    n_slots = len(train_exs[0]['output'])
    if n_slots == 1:
        return [[([by_id[i] for i in ex['input']], by_id[ex['output'][0]]) for ex in train_exs]]
        
    ex0_outs = [by_id[i] for i in train_exs[0]['output']]
    best_perm_slots = None
    best_total_score = -1
    
    for perm in permutations(range(n_slots)):
        total_score = 0
        candidate_slots = []
        
        for slot_idx in range(n_slots):
            slot_data = []
            for ex_idx, ex in enumerate(train_exs):
                inp_items = [by_id[i] for i in ex['input']]
                if ex_idx == 0:
                    out_item = ex0_outs[slot_idx]
                else:
                    out_item = by_id[ex['output'][perm[slot_idx]]]
                slot_data.append((inp_items, out_item))
                
            out_cat = slot_data[0][1]['category']
            _, score = find_best_hypothesis(slot_data, out_cat)
            total_score += score
            candidate_slots.append(slot_data)
            
        if total_score > best_total_score:
            best_total_score = total_score
            best_perm_slots = candidate_slots
            
    return best_perm_slots

def solve_task(task_data):
    train = task_data['train']
    test_input = task_data['test'][0]['input']
    test_items = [by_id[iid] for iid in test_input if iid in by_id]
    
    slots = align_slots(train)
    predictions = set()
    exclude = set(test_input)
    
    for slot_data in slots:
        cat_counts = Counter(out['category'] for _, out in slot_data)
        out_cat = cat_counts.most_common(1)[0][0]
        
        hyp, score = find_best_hypothesis(slot_data, out_cat)
        pred = apply_hypothesis(hyp, test_items, out_cat, exclude)
        
        if pred:
            predictions.add(pred['item_id'])
            exclude.add(pred['item_id'])
            
    return predictions

submission_rows = []
for tid, tdata in tasks.items():
    pred_ids = solve_task(tdata)
    if not pred_ids:
        pred_ids = {catalog[0]['item_id']}
    submission_rows.append((tid, ' '.join(str(x) for x in sorted(pred_ids))))

sub_df = pd.DataFrame(submission_rows, columns=['task_id', 'recommended_item_ids'])
ss_df = pd.read_csv(PUB / 'sample_submission.csv')
sub_df = ss_df[['task_id']].merge(sub_df, on='task_id', how='left')
sub_df['recommended_item_ids'] = sub_df['recommended_item_ids'].fillna(str(catalog[0]['item_id']))

sub_df.to_csv(OUT, index=False)
